# RandLA-Net Using Open3D-ML

https://github.com/isl-org/Open3D-ML?tab=readme-ov-file#semantic-segmentation

In [ ]:
%load_ext autoreload
%autoreload 2

In [5]:
import os
import open3d.ml as _ml3d
import open3d.ml.torch as ml3d


cfg_file = "/home/arthur/Documents/Code/Github/Open3D-ML/ml3d/configs/randlanet_semantickitti.yml"
cfg = _ml3d.utils.Config.load_from_file(cfg_file)

In [7]:
model = ml3d.models.RandLANet(**cfg.model)
cfg.dataset['dataset_path'] = '/media/arthur/HDD/Datasets/Point Clouds/SemanticKitti/raw/'
dataset = ml3d.datasets.SemanticKITTI(cfg.dataset.pop('dataset_path', None), **cfg.dataset)
pipeline = ml3d.pipelines.SemanticSegmentation(model, dataset=dataset, device="gpu", **cfg.pipeline)

In [8]:
# download the weights.
ckpt_folder = "./logs/"
os.makedirs(ckpt_folder, exist_ok=True)
ckpt_path = ckpt_folder + "randlanet_semantickitti_202201071330utc.pth"
randlanet_url = "https://storage.googleapis.com/open3d-releases/model-zoo/randlanet_semantickitti_202201071330utc.pth"
if not os.path.exists(ckpt_path):
    cmd = "wget {} -O {}".format(randlanet_url, ckpt_path)
    os.system(cmd)

--2024-05-13 19:17:58--  https://storage.googleapis.com/open3d-releases/model-zoo/randlanet_semantickitti_202201071330utc.pth
Resolving storage.googleapis.com (storage.googleapis.com)... 2a00:1450:4007:80c::201b, 2a00:1450:4007:80e::201b, 2a00:1450:4007:813::201b, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|2a00:1450:4007:80c::201b|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5101179 (4,9M) [application/octet-stream]
Saving to: ‘./logs/randlanet_semantickitti_202201071330utc.pth’

     0K .......... .......... .......... .......... ..........  1%  479K 10s
    50K .......... .......... .......... .......... ..........  2%  480K 10s
   100K .......... .......... .......... .......... ..........  3% 1,23M 8s
   150K .......... .......... .......... .......... ..........  4% 1,11M 7s
   200K .......... .......... .......... .......... ..........  5% 1,50M 6s
   250K .......... .......... .......... .......... ..........  6% 1,99M 5s
   30

In [9]:
# load the parameters.
pipeline.load_ckpt(ckpt_path=ckpt_path)

In [10]:
test_split = dataset.get_split("test")
data = test_split.get_data(0)

In [11]:
# run inference on a single example.
# returns dict with 'predict_labels' and 'predict_scores'.
result = pipeline.run_inference(data)

test 0/1: 100%|█████████▉| 79834/79845 [00:02<00:00, 30506.56it/s]/home/arthur/miniconda3/envs/open3d/lib/python3.10/site-packages/open3d/_ml3d/torch/modules/metrics/semseg_metric.py:54: RuntimeWarning: Mean of empty slice
  accs.append(np.nanmean(accs))
/home/arthur/miniconda3/envs/open3d/lib/python3.10/site-packages/open3d/_ml3d/torch/modules/metrics/semseg_metric.py:87: RuntimeWarning: Mean of empty slice
  ious.append(np.nanmean(ious))


In [12]:
# evaluate performance on the test set; this will write logs to './logs'.
pipeline.run_test()

test 0/1: 100%|██████████| 79845/79845 [00:19<00:00, 30506.56it/s]